In [1]:
import math
import sys
import time
import numpy as np

rng = np.random.default_rng(42)

def measure(fn, x, repeat=3):
    best = float("inf")
    result = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = fn(x)
        t1 = time.perf_counter()
        best = min(best, t1 - t0)
    return result, best

In [2]:
# Задача 1
def loop_sum(a):
    s = 0.0
    for x in a:
        s += x
    return s


def numpy_sum(a):
    return a.sum()


print(f"{'n':<10} | {'Цикл (мс)':<12} | {'NumPy (мс)':<12} | {'Ускорение':<10}")
print("-" * 52)

for n in [10**5, 10**6, 10**7]:
    a = rng.normal(size=n)
    r_loop, t_loop = measure(loop_sum, a)
    r_numpy, t_numpy = measure(numpy_sum, a)
    print(f"{n:<10} | {t_loop*1000:<12.2f} | {t_numpy*1000:<12.2f} | {t_loop / t_numpy:.1f}x")
    assert np.isclose(r_loop, r_numpy)


n          | Цикл (мс)    | NumPy (мс)   | Ускорение 
----------------------------------------------------
100000     | 24.74        | 0.06         | 394.5x
1000000    | 240.60       | 0.63         | 381.2x
10000000   | 1202.02      | 7.28         | 165.1x


In [3]:
# Задача 2
a = rng.normal(size=10**6)


def loop_mean_std(a):
    s = 0.0
    s_sq = 0.0
    for x in a:
        s += x
        s_sq += x * x
    n = len(a)
    mean = s / n
    std = math.sqrt((s_sq / n) - (mean * mean))
    return mean, std


def numpy_mean_std(a):
    return a.mean(), a.std()


r_loop, t_loop = measure(loop_mean_std, a)
r_numpy, t_numpy = measure(numpy_mean_std, a)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают (1e-12): {np.allclose(r_loop, r_numpy, atol=1e-12, rtol=1e-12)}")


цикл:          252.86 мс
NumPy:           2.53 мс
ускорение: 100.0x
результаты совпадают (1e-12): True


In [4]:
# Задача 3
a = rng.normal(size=10**6)


def loop_transform(a):
    res = [0.0] * len(a)
    for i in range(len(a)):
        x = a[i]
        res[i] = math.sin(x) + x * x
    return res


def numpy_transform(a):
    return np.sin(a) + a**2


r_loop, t_loop = measure(loop_transform, a)
r_numpy, t_numpy = measure(numpy_transform, a)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")


цикл:          361.99 мс
NumPy:          20.62 мс
ускорение: 17.6x
результаты совпадают: True


In [ ]:
# Задача 4
a = rng.normal(size=10**6)

def loop_clip(a):
    res = [0.0] * len(a)
    for i in range(len(a)):
        x = a[i]
        if x < 0:
            res[i] = 0.0
        elif x > 1:
            res[i] = 1.0
        else:
            res[i] = x
    return res

def numpy_clip(a):
    return np.clip(a, 0, 1)

r_loop, t_loop = measure(loop_clip, a)
r_numpy, t_numpy = measure(numpy_clip, a)
changed = np.sum((a < 0) | (a > 1))

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")
print(f"изменено элементов: {changed}")